








































































































































## Import bibliotek

In [1]:
import requests
from bs4 import BeautifulSoup

Potencjalne produkty

32918774
83177636
91869341

## Pobranie z serwisu [Ceneo.pl](https;//Ceneo.pl) opinii o wybranym produkcie

### Wysłanie żądania dostępu do zasobu (strony WWW)


In [2]:
headers = {
    "Cookie":"sv3=1.0_edd5e284-1e04-11f1-8f6d-7d844dec98d7; urdsc=1; userCeneo=ID=3e5d9c2e-d5a8-445a-ad18-85d9a99424c3; __RequestVerificationToken=wFVm9U6qAdyOgHlXyKkTC6juq6IT-pSA8hKPmqgKfLazrHLvwjFC3I2o2CVkbNjRC0zoDF65daikg7eeVSMv1zk6As_7w19szya5ptI-TM41; st2=_gd%3dwww.google.com%2csref%3dhttps%3a%2f%2fwww.google.com%2f%2c_t%3d63908914594%2cencode%3dtrue; ai_user=f7pQ8|2026-03-12T11:16:35.304Z; ai_session=9UJiv|1773314195468|1773315993773; __utmf=0951e77eacc681f9c32401b00333ffe2_Dsgqi6QMc9CtX7buqOpcIw%3D%3D; appType=%7B…ble_cookie=1; _ttp=01KKGW7T7JSWD4KTY9RFT104VN_.tt.1; ttcsid_CNK74OBC77U1PP7E4UR0=1773314238707::YCwGadZDcQ4xjZus9mTI.1.1773315994818.1; ttcsid=1773314238707::z2GVjvtWfOtn7lM29-uc.1.1773315994818.0; __gads=ID=923d0bcddead1efb:T=1773314237:RT=1773314820:S=ALNI_MZjbmxEbPJwrSqwych9DGiJBc2u9w; __gpi=UID=00001379719b9094:T=1773314237:RT=1773314820:S=ALNI_Mb-cOgiKO8IMxrLcJVbKWcX41D5Vw; __eoi=ID=6362c2d756a84f8f:T=1773314237:RT=1773314820:S=AA-AfjYZEU88Cv9HTl3Wm3XpARcQ; nps3=SessionStartTime=1773315992,SurveyId=67",
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:148.0) Gecko/20100101 Firefox/148.0",
    "Host":"www.ceneo.pl"
}

In [3]:
url = "https://www.ceneo.pl/32918774#tab=reviews"
response = requests.get(url)
print(response.status_code)

200


### Parsowanie strony z opiniami

In [4]:
page_dom = BeautifulSoup(response.text,"html.parser") # obiekt klasy BeautifulSoup zwraca document object model, to jest drzewo po którym można chodzić(za pomocą selektorów CSS)
print(page_dom)


<!DOCTYPE html>

<!--[if lt IE 7]> <html class="no-js lt-ie9 lt-ie8 lt-ie7  non-direct ab-inactive" lang="pl" > <![endif]-->
<!--[if IE 7]> <html class="no-js lt-ie9 lt-ie8  non-direct ab-inactive" lang="pl" > <![endif]-->
<!--[if IE 8]> <html class="no-js lt-ie9  non-direct ab-inactive" lang="pl" > <![endif]-->
<!--[if gt IE 8]><!-->
<html class="no-js non-direct ab-inactive" lang="pl">
<!--<![endif]-->
<head>
<meta charset="utf-8"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<link href="https://image.ceneo.pl" rel="preconnect"/>
<link href="https://image.ceneostatic.pl" rel="preconnect"/>
<link href="https://www.google-analytics.com" rel="preconnect"/>
<link href="https://www.googleadservices.com" rel="preconnect"/>
<link href="https://adservice.google.com" rel="preconnect"/>
<link href="https://www.google.pl" rel="preconnect"/>
<link href="https://www.google.com" rel="preconnect"/>
<link href="https://cdn.ampproject.org" rel="preconnect"/>
<title>Urządzenie wielofunkcyj

In [5]:
opinions = page_dom.select("div.js_product-review:not(.user-post--highlight)")
print(type(opinions))
print(len(opinions))

<class 'bs4.element.ResultSet'>
10


In [6]:
opinion = page_dom.select_one("div.js_product-review")
print(type(opinion))

<class 'bs4.element.Tag'>


### Analiza struktury pojedynczej opinii

|składowa|nazwa|selektor|
|--------|-----|--------|
|opinia|opinion|div.js_product-review|div.js_product-review|
|identyfikator|opinion_id|["data-entry-id"]|
|autor|author|span.user-post__author-name|
|treść|content|div.user-post__text|
|ocena|score|span.user-post__score-count|
|rekomendacja|recommendation|span.user-post__author-recomendation > em|
|lista zalet|pros|div.review-feature__item--positive|
|lista wad|cons|div.review-feature__item--negative|
|dla ilu przydatna|thumbs_up|button.vote-yes > span|
|dla ilu nieprzydatna|thumbs_down|button.vote-no > span|
|data zamieszczenia|post_date|span.user-post__published > time:nth-child(1)[datetime]|
|data zakupu|purchase_date|span.user-post__published > time:nth-child(2)[datetime]|

In [ ]:
all_opinions = []

for opinion in opinions:
    opinion_data = {}
    
    # identyfikator opinii
    opinion_data["opinion_id"] = opinion.get("data-entry-id")
    
    # Autor
    opinion_data["author"] = opinion.select_one("span.user-post__author-name").get_text(strip=True)

    # Treść opinii
    opinion_data["content"] = opinion.select_one("div.user-post__text").get_text(strip=True)

    # Ocena
    opinion_data["score"] = opinion.select_one("span.user-post__score-count").get_text(strip=True)
    opinion_data["score"] = float(opinion_data["score"].split("/")[0].replace(",", "."))

    # Rekomendacja (Polecam/Nie polecam)
    try:
        opinion_data["recommendation"] = opinion.select_one(".user-post__author-recomendation > em").get_text(strip=True)
    except AttributeError:
        opinion_data["recommendation"] = None
    
    # Zalety
    opinion_data["pros"] = [p.get_text(strip=True) for p in opinion.select("div.review-feature__item--npositive")]
    
    # Wady
    opinion_data["cons"] = [c.get_text(strip=True) for c in opinion.select("div.review-feature__item--egative")]
    
    # Dla ilu osób przydatna
    opinion_data["useful"] = int(opinion.select_one("button.vote-yes > span").get_text(strip=True))
    
    # Dla ilu osób nieprzydatna
    opinion_data["unuseful"] = int(opinion.select_one("button.vote-no > span").get_text(strip=True))
    
    # Data zamieszczenia
    opinion_data["published_date"] = opinion.select_one("span.user-post__published > time:nth-child(1)").get("datetime")
    
    # Data zakupu / potwierdzony zakup
    try:
        opinion_data["purchase_date"] = opinion.select_one("span.user-post__published > time:nth-child(2)").get("datetime")
    except AttributeError:
        opinion_data["purchase_date"] = None
    
    all_opinions.append(opinion_data)
    
# Wyświetlenie zebranych opinii
for single_opinion in all_opinions:
    print(single_opinion)

{'opinion_id': '14163661', 'author': 'r...t', 'content': 'Bardzo jestem zadowolony z zakupu funkcjonalnej i przydatnej w pracy mobilnej drukarki. Zdecydowanie zasługuje ona na same plusy i w ramach ceny, jaką zapłaciłem, uważam, że posiada same zalety. Szukałem takiego produktu, znalazłem produkt i jestem zadowolony z produktu. Polecam zdecydowanie wszystkim, którzy mają podobne potrzeby korzystania z drukarki, przemieszczając się. Dodałbym jeszcze, jako jedną z ważniejszych zalet, jej wagę i gabaryty, a więc, możliwość zapakowania do małego plecaczka, razem z laptopem.', 'score': 5.0, 'recommendation': 'Polecam', 'pros': [], 'cons': ['często żle pobiera papier', 'dodatkowy zbiornik na zużyty atrament', 'głośność pracy', 'jakość wydruku', 'lekka', 'przenośna', 'szybkość wydruku', 'wielkość'], 'useful': 0, 'unuseful': 0, 'published_date': '2021-03-23 16:14:28', 'purchase_date': '2020-07-17 23:59:25'}
{'opinion_id': '19353234', 'author': 'e...k', 'content': 'Mała kompaktowa drukarka. Jak

In [20]:
print(len(opinions))
print(len(all_opinions))

10
10


## Ładna  tabelka

In [16]:
import pandas as pd

In [17]:
df = pd.DataFrame(all_opinions)
df

,opinion_id,author,content,score,recommendation,pros,cons,useful,unuseful,published_date,purchase_date
0,14163661,r...t,Bardzo jestem zadowolony z zakupu funkcjonalne...,5.0,Polecam,[],"[często żle pobiera papier, dodatkowy zbiornik...",0,0,2021-03-23 16:14:28,2020-07-17 23:59:25
1,19353234,e...k,Mała kompaktowa drukarka. Jakość wydruku bardz...,5.0,Polecam,[],[],0,0,2025-01-07 10:17:01,2024-12-31 11:40:48
2,13777122,Kreoger,Zalety:\n- kompaktowa i elegancka\n- jakość dr...,5.0,Polecam,[],"[dodatkowy zbiornik na zużyty atrament, głośno...",0,0,2021-01-15 10:01:23,2021-01-12 10:34:23
3,18898199,d...e,"Mała drukarka przenośna A4, kolorowa.\nIdealna...",4.5,Polecam,[],"[dodatkowy zbiornik na zużyty atrament, głośno...",0,0,2024-08-21 11:18:20,2024-07-23 19:19:20
4,14051988,t...8,Super drukareczka mieści się w torbie razem z ...,5.0,Polecam,[],"[często żle pobiera papier, dodatkowy zbiornik...",0,0,2021-03-03 19:43:07,2021-02-01 10:33:58
5,6626391,Użytkownik najmniejszej drukarki,Jestem z drukarki zadowolony. To początek zoba...,4.5,Polecam,[szybkość wydruku],[],1,0,2018-02-22 11:35:43,2018-02-02 19:31:47
6,13239320,Skywalker Luk,"Idealna w podróży, nie drukuje za szybko ale ...",5.0,Polecam,[],"[głośność pracy, jakość wydruku, przenośna]",0,0,2020-10-19 22:41:59,2020-10-01 00:07:55
7,8624863,Dawid,"Wyśmienita, maleńka, kolorowa, polecam.",5.0,Polecam,[],"[głośność pracy, jakość wydruku, lekka, przeno...",1,1,2018-11-28 10:16:31,2018-11-23 17:05:52
8,18249412,G...A,szkoda że nie występuje w wersji proszkowej/ka...,5.0,Polecam,[głośność pracy],"[jakość wydruku, przenośna, szybkość wydruku]",0,0,2023-12-27 21:34:35,2023-12-21 09:28:14
9,16424063,Użytkownik Ceneo,Chciałam drukarkę a dostałam tusz do drukarki ...,1.0,Nie polecam,[],[],0,7,2022-08-16 14:31:46,2022-08-12 14:30:43


In [27]:
page = 1
next = True

while next:
    url = (f"https://www.ceneo.pl/32918774/opinie-{page}")
    response = requests.get(url)
    print(f"{url} => {response.status_code}")
    page_dom = BeautifulSoup(response.text, "html.parser")
    next = True if page_dom.select_one("button.pagination__next") else False
    page += 1

https://www.ceneo.pl/32918774/opinie-1 => 200
https://www.ceneo.pl/32918774/opinie-2 => 200
